In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install streamlit transformers torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 83.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 121.8 MB/s eta 0:00:00


In [ ]:
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

--2026-04-22 04:54:50--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
Resolving github.com (github.com)... 140.82.112.3
Connecting to github.com (github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2026.3.0/cloudflared-linux-amd64.deb [following]
--2026-04-22 04:54:50--  https://github.com/cloudflare/cloudflared/releases/download/2026.3.0/cloudflared-linux-amd64.deb
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/ec689fe1-d727-4ebd-bbc3-5967730ab54e?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-04-22T05%3A46%3A05Z&rscd=attachment%3B+filename%3Dcloudflared-linux-amd64.deb&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&

In [ ]:
%%writefile app.py
import streamlit as st
from transformers import pipeline

# --- PAGE CONFIG ---
st.set_page_config(page_title="EmpathyBot AI", page_icon="🤖")

# --- MODEL LOADING (Cached) ---
@st.cache_resource
def load_model():
    # Loading the GoEmotions model for human-like detection
    return pipeline("text-classification",
                    model="SamLowe/roberta-base-go_emotions",
                    top_k=1)

classifier = load_model()

# --- EMOTION TO RESPONSE MAPPING ---
EMOTION_MAP = {
    "admiration": "That is so kind of you! It really warms my circuits.",
    "amusement": "Haha! That's genuinely funny.",
    "anger": "I can see you're really frustrated. I'm listening—tell me how I can fix this.",
    "annoyance": "I'm sorry, I know how irritating that can be. Let's get it sorted.",
    "caring": "That's very thoughtful of you. I appreciate the kindness.",
    "confusion": "I might have tripped over my own logic there. Let me try to explain it better.",
    "disappointment": "I'm truly sorry to hear that. I never want to let you down.",
    "gratitude": "You're very welcome! I'm just happy to be of service.",
    "joy": "That’s amazing news! I’m genuinely happy for you.",
    "optimism": "I love that energy! Let's keep that momentum going.",
    "sadness": "I'm so sorry you're feeling down. I'm here if you need to talk it through.",
    "fear": "That sounds really stressful. Take a breath; I'm here to help you navigate this.",
    "neutral": "I hear you. How can I help you move forward with this?"
}

# --- APP UI ---
st.title("🤖 Empathetic Chatbot")
st.caption("A chatbot that understands your feelings and responds like a human.")

# Initialize chat history
if "messages" not in st.session_state:
    st.session_state.messages = []

# Display chat history from session state
for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

# --- CHAT LOGIC ---
if prompt := st.chat_input("How are you feeling today?"):
    # 1. Display user message
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    # 2. Analyze Sentiment
    prediction = classifier(prompt)[0][0]
    emotion = prediction['label']
    score = prediction['score']

    # 3. Generate Human-like Response
    response_text = EMOTION_MAP.get(emotion, "I hear you, and I'm following what you're saying.")

    # Add intensity layer
    if score > 0.8 and emotion in ['anger', 'sadness', 'fear']:
        response_text = f"I can really feel the weight of this. {response_text}"

    # 4. Display Bot response
    with st.chat_message("assistant"):
        st.markdown(response_text)
        # Subtle "Emotional Intelligence" badge
        st.caption(f"Detected Emotion: {emotion.capitalize()} ({round(score*100)}% confidence)")

    st.session_state.messages.append({"role": "assistant", "content": response_text})

Overwriting app.py


In [ ]:
import subprocess
import time

# 1. Kill any old sessions
!pkill streamlit
!pkill cloudflared

# 2. Start the Streamlit app in the background
with open("streamlit_logs.txt", "w") as f:
    subprocess.Popen(["streamlit", "run", "app.py"], stdout=f, stderr=f)

# 3. Give the AI model time to load into RAM
print("Loading the AI model... please wait ~15 seconds.")
time.sleep(15)

# 4. Start the Cloudflare Tunnel
print("Creating your secure link...")
!cloudflared tunnel --url http://localhost:8501

Loading the AI model... please wait ~15 seconds.
Creating your secure link...
2026-04-22T04:56:07Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-04-22T04:56:07Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-04-22T04:56:09Z INF +--------------------------------------------------------------------------------------------+
2026-04-22T04:56:09Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
202